# OAI Screening Images — Download, Extract & Save to Google Drive

Downloads **OAIScreeningImages** from the NIMH Data Archive, extracts the ZIP, and saves **only the final X-ray images** to Google Drive using `nda-tools`.

**What this does:**
1. Downloads `P001.zip` (~40 GB) to Colab's local SSD (`/content/`)
2. Extracts it locally on SSD
3. Copies only the raw X-ray images (~3–4 GB) to Drive — ZIP and thumbnails are discarded

**Requirements:**
- Google Drive with ≥ 6 GB free
- NDA username and password (nda.nih.gov)
- Approved Data Access Request (DAR #24657)

**Expected time:** 1–3 hours depending on network speed.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import shutil
from pathlib import Path

DEST = '/content/drive/MyDrive/oai/screening_images'
dest = Path(DEST)

confirm = input(
    'WARNING: This permanently deletes ALL contents of:\n'
    f'  {DEST}\n\n'
    'Type  yes  to confirm, anything else to cancel: '
).strip().lower()

if confirm != 'yes':
    print('Cancelled — nothing deleted.')
else:
    if dest.exists():
        all_files = [f for f in dest.rglob('*') if f.is_file()]
        total_gb = sum(f.stat().st_size for f in all_files) / 1e9
        print(f'Deleting {len(all_files):,} files ({total_gb:.2f} GB) ...')
        shutil.rmtree(str(dest))
    else:
        print('Directory does not exist — nothing to delete.')
    dest.mkdir(parents=True, exist_ok=True)
    print(f'Clean directory created: {DEST}')
    print('Run the cells below (skip this one) to re-download.')


  /content/drive/MyDrive/oai/screening_images

Type  yes  to confirm, anything else to cancel: yes
Deleting 0 files (0.00 GB) ...
Clean directory created: /content/drive/MyDrive/oai/screening_images
Run the cells below (skip this one) to re-download.


In [4]:
import shutil

total_local, used_local, free_local = shutil.disk_usage('/content')
total_drive, used_drive, free_drive = shutil.disk_usage('/content/drive/MyDrive')

print(f"Colab local SSD free : {free_local / 1e9:.1f} GB  (needs ≥ 40 GB for ZIP)")
print(f"Google Drive free    : {free_drive / 1e9:.1f} GB  (needs ≥ 6 GB for images)")

ok = True
if free_local < 40e9:
    print("WARNING: Less than 40 GB free on local SSD — ZIP download may fail.")
    ok = False
if free_drive < 6e9:
    print("WARNING: Less than 6 GB free on Drive — image extraction may fail.")
    ok = False
if ok:
    print("Space OK — ready to download.")


Colab local SSD free : 93.7 GB  (needs ≥ 40 GB for ZIP)
Google Drive free    : 45.4 GB  (needs ≥ 6 GB for images)
Space OK — ready to download.


In [5]:
!pip install nda-tools keyrings.alt --quiet
print("Done.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.5/105.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 61.0 MB/s eta 0:00:00
Done.


In [6]:
import getpass, keyring, os

os.environ['PYTHON_KEYRING_BACKEND'] = 'keyrings.alt.file.PlaintextKeyring'

nda_username = input("NDA username: ").strip()
nda_password = getpass.getpass("NDA password (not your login.gov password): ")

keyring.set_password('nda-tools', nda_username, nda_password)
print(f"Credentials stored for: {nda_username}")


NDA username: samihamrouni282000
NDA password (not your login.gov password): ··········
Credentials stored for: samihamrouni282000


In [ ]:
import os

DEST = '/content/drive/MyDrive/Master Thesis/oai/screening_images'
os.makedirs(DEST, exist_ok=True)
print(f"Destination: {DEST}")


Destination: /content/drive/MyDrive/oai/screening_images


In [8]:


PACKAGE_ID = input("Personal package ID: ").strip()
print(f"Package: {PACKAGE_ID}")


Personal package ID: 1245974
Package: 1245974


In [9]:
import subprocess, sys, os, threading
from pathlib import Path

os.environ['PYTHON_KEYRING_BACKEND'] = 'keyrings.alt.file.PlaintextKeyring'

LOCAL_DL = '/content/oai_staging'
os.makedirs(LOCAL_DL, exist_ok=True)

stop_monitor = threading.Event()

def monitor_progress(path, label, interval=60):
    while not stop_monitor.wait(interval):
        try:
            files = [f for f in Path(path).rglob('*') if f.is_file()]
            total_gb = sum(f.stat().st_size for f in files) / 1e9
            print(f"  [{label}] {len(files):,} files | {total_gb:.2f} GB", flush=True)
        except Exception:
            pass

threading.Thread(target=monitor_progress, args=(LOCAL_DL, 'download'), daemon=True).start()
print(f"Stage 1 — downloading to local SSD: {LOCAL_DL}\n")

proc = subprocess.Popen(
    [sys.executable, '-m', 'NDATools.clientscripts.downloadcmd',
     '-dp', PACKAGE_ID, '-d', LOCAL_DL, '-wt', '10', '-u', nda_username],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    cwd=LOCAL_DL
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
stop_monitor.set()

print(f'\nDownload exit code: {proc.returncode}')
if proc.returncode != 0:
    raise RuntimeError('Download failed — check output above.')

zip_path = next(Path(LOCAL_DL).rglob('*.zip'), None)
if zip_path is None:
    raise FileNotFoundError(f'No ZIP found in {LOCAL_DL}')
print(f'ZIP ready: {zip_path}  ({zip_path.stat().st_size/1e9:.2f} GB)')


Stage 1 — downloading to local SSD: /content/oai_staging



                  Running NDA Tools Version 0.7.0

Using configuration file from /root/.NDATools/settings.cfg
proceeding as NDA user: samihamrouni282000

Getting Package Information...

Package-id: 1245974
Name: OAIScreeningImages
Has associated files?: Yes
Number of files in package: 6
Total Package Size: 44.19GB

Starting download: /root/NDA/nda-tools/downloadcmd/packages/1245974/package_file_metadata_1245974.txt.gz.partial
Completed download /root/NDA/nda-tools/downloadcmd/packages/1245974/package_file_metadata_1245974.txt.gz

S3 links for files that failed to download will be written out to /root/NDA/nda-tools/downloadcmd/logs/failed_s3_links_file_20260510T1426053gl228wa.csv. You can attempt to download these files later by running: 
	downloadcmd -dp 1245974 -u samihamrouni282000 -d /content/oai_staging -wt 10 -t "/root/NDA/nda-tools/downloadcmd/logs/failed_s3_links_file_20260510T1426053gl228wa.csv"


Beginning download of

In [10]:
import subprocess, shutil
from pathlib import Path

LOCAL_DL = '/content/oai_staging'
zip_path = next(Path(LOCAL_DL).rglob('*.zip'), None)
if zip_path is None:
    raise FileNotFoundError(f'No ZIP found in {LOCAL_DL} — run Stage 1 first.')

dest_path = Path(DEST)
dest_path.mkdir(parents=True, exist_ok=True)

zip_gb = zip_path.stat().st_size / 1e9
print(f'Stage 2 — extracting {zip_path.name} ({zip_gb:.2f} GB) directly to Drive: {DEST}')
print('Thumbnails (*.jpg) excluded — only raw X-ray files saved to Drive.\n')

result = subprocess.run(
    ['unzip', '-n',
     str(zip_path),
     '-x', '*.jpg',
     '-d', str(dest_path)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
lines = result.stdout.strip().splitlines()
if len(lines) > 40:
    print(f'[...{len(lines)-40} lines omitted...]')
print('\n'.join(lines[-40:]))

if result.returncode not in (0, 1):
    raise RuntimeError(f'unzip failed with code {result.returncode}')
print('Extraction complete.')

shutil.rmtree(LOCAL_DL, ignore_errors=True)
print(f'Local SSD freed: {zip_gb:.2f} GB removed from {LOCAL_DL}')
print(f'\nDone — X-ray images are in Drive: {DEST}')


Stage 2 — extracting P001.zip (47.44 GB) directly to Drive: /content/drive/MyDrive/oai/screening_images
Thumbnails (*.jpg) excluded — only raw X-ray files saved to Drive.

[...4757 lines omitted...]
  inflating: /content/drive/MyDrive/oai/screening_images/0.C.2/9702463/20040422/00034504/001  
  inflating: /content/drive/MyDrive/oai/screening_images/0.C.2/9816968/20040603/00069904/001  
  inflating: /content/drive/MyDrive/oai/screening_images/0.C.2/9746894/20040826/00230804/001  
  inflating: /content/drive/MyDrive/oai/screening_images/0.C.2/9979265/20041202/00409804/001  
  inflating: /content/drive/MyDrive/oai/screening_images/0.C.2/9318127/20050324/00658804/001  
  inflating: /content/drive/MyDrive/oai/screening_images/0.C.2/9925985/20041101/00371103/001  
  inflating: /content/drive/MyDrive/oai/screening_images/0.C.2/9871240/20040707/00113004/001  
  inflating: /content/drive/MyDrive/oai/screening_images/0.C.2/9684211/20041118/00405203/001  
  inflating: /content/drive/MyDrive/oai/s

In [ ]:
from pathlib import Path
from collections import Counter

DEST = '/content/drive/MyDrive/Master Thesis/oai/screening_images'
dest = Path(DEST)

if not dest.exists():
    print(f'Directory not found: {DEST}')
else:
    all_files = [f for f in dest.rglob('*') if f.is_file()]
    total_gb  = sum(f.stat().st_size for f in all_files) / 1e9
    print(f'=== {DEST} ===')
    print(f'Total : {len(all_files):,} files  |  {total_gb:.2f} GB\n')

    print('Top-level contents:')
    for item in sorted(dest.iterdir()):
        if item.is_dir():
            n  = sum(1 for f in item.rglob('*') if f.is_file())
            sz = sum(f.stat().st_size for f in item.rglob('*') if f.is_file()) / 1e9
            print(f'  [DIR ] {item.name}/  — {n:,} files, {sz:.2f} GB')
        else:
            print(f'  [FILE] {item.name}  — {item.stat().st_size/1e9:.3f} GB')

    ext_counts = Counter(f.suffix.lower() for f in all_files)
    print('\nFile types:')
    for ext, count in ext_counts.most_common():
        label = (ext if ext else '(no ext) ← X-ray DICOMs')
        sz = sum(f.stat().st_size for f in all_files if f.suffix.lower() == ext) / 1e9
        print(f'  {label:35s}: {count:6,} files  ({sz:.2f} GB)')

    xray_files  = [f for f in all_files if f.suffix == '']
    xray_gb     = sum(f.stat().st_size for f in xray_files) / 1e9
    subject_ids = set()
    for f in xray_files:
        parts = f.relative_to(dest).parts
        if len(parts) >= 2:
            subject_ids.add(parts[1])

    print(f'\nX-ray images   : {len(xray_files):,}  ({xray_gb:.2f} GB)')
    print(f'Unique subjects: {len(subject_ids):,}')
    if xray_files:
        sizes = [f.stat().st_size / 1e6 for f in xray_files]
        print(f'Size range     : {min(sizes):.1f} – {max(sizes):.1f} MB  (avg {sum(sizes)/len(sizes):.1f} MB)')
        print('\nSample paths:')
        for f in sorted(xray_files)[:5]:
            print(f'  {f.relative_to(dest)}  ({f.stat().st_size/1e6:.1f} MB)')

    leftover_zips = list(dest.rglob('*.zip')) + list(dest.rglob('*.partial'))
    if leftover_zips:
        lz_gb = sum(f.stat().st_size for f in leftover_zips) / 1e9
        print(f'\nWARNING: {len(leftover_zips)} leftover ZIP/partial file(s) ({lz_gb:.2f} GB) wasting Drive space:')
        for f in leftover_zips:
            print(f'  {f.relative_to(dest)}')
        print('  → Re-run Stage 2 to delete them, or run:')
        print(f'    import os; [os.remove(str(f)) for f in {[str(f) for f in leftover_zips]}]')

    print('\n=== Status ===')
    if len(xray_files) == 0:
        print('No X-ray images — run Stage 2 (cell 10).')
    elif leftover_zips:
        print(f'{len(xray_files):,} images OK, but ZIP still on Drive eating space — re-run Stage 2.')
    elif len(xray_files) < 1000:
        print(f'Only {len(xray_files):,} images — may be incomplete. Re-run Stage 2.')
    else:
        print(f'OK — {len(xray_files):,} images ({xray_gb:.2f} GB). Ready for oai_labels.ipynb.')


=== /content/drive/MyDrive/oai/screening_images ===
Total : 4,796 files  |  78.95 GB

Top-level contents:
  [DIR ] 0.C.2/  — 2,686 files, 44.70 GB
  [DIR ] 0.E.1/  — 2,110 files, 34.25 GB

File types:
  (no ext) ← X-ray DICOMs            :  4,796 files  (78.95 GB)

X-ray images   : 4,796  (78.95 GB)
Unique subjects: 4,795
Size range     : 7.2 – 33.9 MB  (avg 16.5 MB)

Sample paths:
  0.C.2/9000296/20040729/00175004/001  (10.2 MB)
  0.C.2/9000798/20040802/00162704/001  (10.2 MB)
  0.C.2/9001400/20041202/00410004/001  (13.1 MB)
  0.C.2/9001695/20041203/00422803/001  (29.6 MB)
  0.C.2/9001897/20050112/00464604/001  (9.9 MB)

=== Status ===
OK — 4,796 images (78.95 GB). Ready for oai_labels.ipynb.
